In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import warnings

# We will ignore standard warnings to keep our notebook clean
warnings.filterwarnings('ignore')

In [4]:
file_path = "../../dataset/fraud/output/synthetic_invoice_dataset.csv"
df = pd.read_csv(file_path)
df = df.dropna()
# 2. Display the shape of the dataset
print("Dataset Shape:")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("-" * 30)

# 3. Display the first 3 rows to see what the data looks like
print("\nFirst 3 rows of the dataset:")
display(df.head(3))

# 4. Check the data types of every column
print("\nData Types:")
print(df.dtypes)

Dataset Shape:
Rows: 14760
Columns: 19
------------------------------

First 3 rows of the dataset:


,invoice_number,vendor_id,vendor_name,vendor_category,country,city,po_number,department,invoice_date,due_date,payment_terms,currency,amount,tax_rate,tax_amount,invoice_status,created_timestamp,is_fraud,fraud_type
0,INV-901781-7501,VEND-99FCF6C7,Srivastava LLC,Electronics,India,Hazaribagh,PO-7A1835AA,R&D,2024-11-21,2025-02-19,90,INR,94194.97,0.5091,47954.66,PAID,2024-11-21T20:17:59,1,TAX
1,INV-900858-9865,VEND-34425F7F,"Gera, Baral and Kota",Furniture,India,Bahraich,PO-410C53CE,Procurement,2023-09-07,2023-11-06,60,INR,54163.61,0.1383,7490.83,PAID,2023-09-07T04:05:47,1,PO_MISMATCH
2,INV-002900-847B,VEND-FC45B8AB,Murthy-Sahota Enterprises,Transportation,China,Haridwar,PO-958632C4,Facilities,2023-08-14,2023-11-12,90,CNY,4470.75,0.1047,468.09,OVERDUE,2023-08-14T11:34:16,0,NONE



Data Types:
invoice_number           str
vendor_id                str
vendor_name              str
vendor_category          str
country                  str
city                     str
po_number                str
department               str
invoice_date             str
due_date                 str
payment_terms          int64
currency                 str
amount               float64
tax_rate             float64
tax_amount           float64
invoice_status           str
created_timestamp        str
is_fraud               int64
fraud_type               str
dtype: object


In [5]:
# 1. Convert text dates into actual Pandas datetime objects
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['due_date'] = pd.to_datetime(df['due_date'])

# 2. Extract standard date components from invoice_date
df['invoice_year'] = df['invoice_date'].dt.year
df['invoice_month'] = df['invoice_date'].dt.month
df['invoice_day'] = df['invoice_date'].dt.day

# dt.weekday returns 0 for Monday and 6 for Sunday
df['invoice_weekday'] = df['invoice_date'].dt.weekday

# dt.quarter returns 1 to 4 depending on the time of year
df['invoice_quarter'] = df['invoice_date'].dt.quarter

# 3. Create a flag (1 or 0) to check if the invoice was generated on a weekend
# Weekdays 5 and 6 are Saturday and Sunday
df['is_weekend'] = (df['invoice_weekday'] >= 5).astype(int)

# 4. Calculate duration features (time differences)
# How many days between the invoice date and when it is due?
df['days_until_due'] = (df['due_date'] - df['invoice_date']).dt.days

# Invoice age: How many days ago was this invoice created?
# We will assume 'today' is the most recent date in the dataset to simulate reality
current_date = df['invoice_date'].max()
df['invoice_age'] = (current_date - df['invoice_date']).dt.days

print("Date features successfully created!")

Date features successfully created!


In [6]:
# 1. Log Transformation of the amount
# We use np.log1p (log(1 + x)) instead of np.log to avoid errors if amount is 0
df['amount_log'] = np.log1p(df['amount'])

# 2. Calculate the effective tax percentage
# Sometimes fraudsters invent random tax amounts. Let's see the actual percentage.
# We add a tiny number (0.0001) to the denominator to prevent division by zero errors
df['tax_percentage'] = (df['tax_amount'] / (df['amount'] + 0.0001)) * 100

# 3. Amount per day of the payment term
# This shows how "intense" the payment is. A huge amount due in 1 day is suspicious.
# Again, adding 0.0001 to avoid dividing by zero if days_until_due is 0
df['amount_per_day'] = df['amount'] / (df['days_until_due'] + 0.0001)

# 4. Rounded amount flag
# Fraudsters often invent neat, rounded numbers like $5000.00 instead of real numbers like $4932.17
# We check if the amount modulo 100 is 0 (meaning it's perfectly divisible by 100)
df['rounded_amount_flag'] = (df['amount'] % 100 == 0).astype(int)

# 5. High-value invoice flag
# Let's flag any invoice that is in the top 5% of all invoice amounts
top_5_percent_threshold = df['amount'].quantile(0.95)
df['high_value_invoice'] = (df['amount'] >= top_5_percent_threshold).astype(int)

print("Amount features successfully created!")

Amount features successfully created!


In [7]:
# 1. Total number of invoices sent by this vendor
df['vendor_invoice_count'] = df.groupby('vendor_id')['invoice_number'].transform('count')

# 2. Total money spent on this vendor across the whole dataset
df['vendor_total_spend'] = df.groupby('vendor_id')['amount'].transform('sum')

# 3. The average (mean) invoice amount for this vendor
df['vendor_average_amount'] = df.groupby('vendor_id')['amount'].transform('mean')

# 4. The middle value (median) invoice amount for this vendor
df['vendor_median_amount'] = df.groupby('vendor_id')['amount'].transform('median')

# 5. The largest and smallest invoice ever sent by this vendor
df['vendor_max_amount'] = df.groupby('vendor_id')['amount'].transform('max')
df['vendor_min_amount'] = df.groupby('vendor_id')['amount'].transform('min')

# 6. Standard Deviation (how much their amounts fluctuate)
# fillna(0) ensures that if a vendor only has 1 invoice (std is undefined), it becomes 0
df['vendor_std_amount'] = df.groupby('vendor_id')['amount'].transform('std').fillna(0)

# 7. Unique departments this vendor bills to
# A fraudster vendor might try billing multiple random departments
df['vendor_unique_departments'] = df.groupby('vendor_id')['department'].transform('nunique')

print("Vendor behavior features successfully created!")

Vendor behavior features successfully created!


In [8]:
# 1. First, we MUST sort the data chronologically for each vendor
# If we don't sort, 'previous' row means nothing!
df = df.sort_values(by=['vendor_id', 'invoice_date']).reset_index(drop=True)

# 2. Days since this vendor's last invoice
# diff() calculates the difference between current row and the previous row
df['days_since_last_invoice'] = df.groupby('vendor_id')['invoice_date'].diff().dt.days
df['days_since_last_invoice'] = df['days_since_last_invoice'].fillna(-1) # -1 means this is their first invoice

# 3. What was the amount of their previous invoice?
# shift(1) simply pulls the value from 1 row above
df['previous_invoice_amount'] = df.groupby('vendor_id')['amount'].shift(1).fillna(0)

# 4. Difference and Ratio compared to their previous invoice
df['amount_difference'] = df['amount'] - df['previous_invoice_amount']
df['amount_ratio'] = df['amount'] / (df['previous_invoice_amount'] + 0.0001)

# 5. Rolling Averages
# We look at the last 3 and 5 invoices for this specific vendor and take the average
# reset_index(level=0, drop=True) cleans up the format so we can attach it directly to our DataFrame
df['rolling_average_3'] = df.groupby('vendor_id')['amount'].rolling(window=3, min_periods=1).mean().reset_index(level=0, drop=True)
df['rolling_average_5'] = df.groupby('vendor_id')['amount'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)

# 6. Rolling Standard Deviation (Volatility over the last 3 invoices)
df['rolling_std'] = df.groupby('vendor_id')['amount'].rolling(window=3, min_periods=1).std().reset_index(level=0, drop=True).fillna(0)

print("Historical rolling features successfully created!")

Historical rolling features successfully created!


In [9]:
# We use the exact same groupby and transform logic as we did for vendors

# Total number of invoices processed by the department
df['department_invoice_count'] = df.groupby('department')['invoice_number'].transform('count')

# The average invoice amount for this department
df['department_average_amount'] = df.groupby('department')['amount'].transform('mean')

# Total money spent by this department
df['department_total_spend'] = df.groupby('department')['amount'].transform('sum')

print("Department features successfully created!")

Department features successfully created!


In [10]:
# 1. Encode payment terms into payment_days
# Since payment_terms is already an integer in our dataset (e.g., 30 for Net-30)
# we will just assign it to a clearly named variable.
df['payment_days'] = df['payment_terms']

# 2. Create Late Payment Risk
# If an invoice is for a huge amount, but has an aggressively short payment window, 
# it carries higher risk. We divide amount by the payment days.
df['late_payment_risk'] = df['amount'] / (df['payment_days'] + 1)

print("Payment features successfully created!")

Payment features successfully created!


In [11]:
# Number of invoices in this specific vendor category
df['category_invoice_count'] = df.groupby('vendor_category')['invoice_number'].transform('count')

# Average amount spent per invoice in this category
df['category_average_amount'] = df.groupby('vendor_category')['amount'].transform('mean')

# Median amount spent per invoice in this category
df['category_median_amount'] = df.groupby('vendor_category')['amount'].transform('median')

# Total money spent in this category across the entire dataset
df['category_spend'] = df.groupby('vendor_category')['amount'].transform('sum')

print("Category features successfully created!")

Category features successfully created!


In [12]:
# 1. Amount vs Vendor Average
# This shows how many times larger the current invoice is compared to the vendor's normal bill
df['amount_vs_vendor_average'] = df['amount'] / (df['vendor_average_amount'] + 0.0001)

# 2. Amount above average flag
# 1 if the amount is strictly higher than their usual average, 0 if it is lower
df['amount_above_average'] = (df['amount'] > df['vendor_average_amount']).astype(int)

# 3. Duplicate Invoice Flag
# Fraudsters often submit the exact same invoice twice hoping both get paid.
# We mark rows as duplicates if the vendor_id, amount, and invoice_date are identical.
# keep=False ensures BOTH the original and the copy are flagged as duplicates.
df['duplicate_invoice_flag'] = df.duplicated(subset=['vendor_id', 'amount', 'invoice_date'], keep=False).astype(int)

# 4. Duplicate PO Flag
# Purchase Orders (POs) should ideally be unique or rare per invoice.
# We ignore missing values (NaN) so we don't accidentally flag all nulls as duplicates.
df['duplicate_po_flag'] = df.duplicated(subset=['po_number'], keep=False)
df.loc[df['po_number'].isna(), 'duplicate_po_flag'] = False
df['duplicate_po_flag'] = df['duplicate_po_flag'].astype(int)

# 5. Tax Flags
# Let's say a tax rate over 20% is unusually high, and below 1% is suspiciously low
df['high_tax_flag'] = (df['tax_rate'] > 0.20).astype(int)
df['low_tax_flag'] = (df['tax_rate'] < 0.01).astype(int)

# 6. New Vendor Flag
# If a vendor only has 1 invoice in the entire database, they are new/untested.
df['new_vendor_flag'] = (df['vendor_invoice_count'] == 1).astype(int)

# 7. Weekend Invoice Flag
# We already created 'is_weekend' in Section 2, but we will rename it for consistency
df['weekend_invoice_flag'] = df['is_weekend']

print("Fraud-specific features successfully created!")

Fraud-specific features successfully created!


In [13]:
# 1. Create a list of all the text columns we want to convert to numbers
categorical_columns = ['vendor_category', 'country', 'city', 'department', 'currency']

# 2. Initialize the LabelEncoder tool
encoder = LabelEncoder()

# 3. Loop through every column and transform it
for col in categorical_columns:
    # fit_transform learns the mapping and applies it at the same time
    df[col] = encoder.fit_transform(df[col].astype(str))

print("Categorical features successfully encoded into numbers!")

Categorical features successfully encoded into numbers!


In [15]:
# Create a list of useless or cheating columns
columns_to_drop = [
    'invoice_number', 
    'vendor_id', 
    'vendor_name', 
    'po_number', 
    'created_timestamp', 
    'invoice_date', 
    'due_date', 
    'invoice_status', 
    'fraud_type',   # Target leakage!
    'is_weekend'    # We duplicated this into weekend_invoice_flag
]

# Drop them from the DataFrame
# We only use 'columns=' here. No need for 'axis=1' anymore!
df_final = df.drop(columns=columns_to_drop)

print(f"Removed {len(columns_to_drop)} columns.")

Removed 10 columns.


In [16]:
# 1. Create X (all features except the answer)
X = df_final.drop(columns=['is_fraud'])

# 2. Create y (only the answer column)
y = df_final['is_fraud']

# 3. Show our final feature names
print("Final Feature List:")
print(list(X.columns))
print("-" * 30)

# 4. Show the number of features
print(f"Total Number of Features: {X.shape[1]}")
print("-" * 30)

# 5. Display the beautiful, ML-ready dataframe
display(X.head())

Final Feature List:
['vendor_category', 'country', 'city', 'department', 'payment_terms', 'currency', 'amount', 'tax_rate', 'tax_amount', 'invoice_year', 'invoice_month', 'invoice_day', 'invoice_weekday', 'invoice_quarter', 'days_until_due', 'invoice_age', 'amount_log', 'tax_percentage', 'amount_per_day', 'rounded_amount_flag', 'high_value_invoice', 'vendor_invoice_count', 'vendor_total_spend', 'vendor_average_amount', 'vendor_median_amount', 'vendor_max_amount', 'vendor_min_amount', 'vendor_std_amount', 'vendor_unique_departments', 'days_since_last_invoice', 'previous_invoice_amount', 'amount_difference', 'amount_ratio', 'rolling_average_3', 'rolling_average_5', 'rolling_std', 'department_invoice_count', 'department_average_amount', 'department_total_spend', 'payment_days', 'late_payment_risk', 'category_invoice_count', 'category_average_amount', 'category_median_amount', 'category_spend', 'amount_vs_vendor_average', 'amount_above_average', 'duplicate_invoice_flag', 'duplicate_po_flag

,vendor_category,country,city,department,payment_terms,currency,amount,tax_rate,tax_amount,invoice_year,...,category_median_amount,category_spend,amount_vs_vendor_average,amount_above_average,duplicate_invoice_flag,duplicate_po_flag,high_tax_flag,low_tax_flag,new_vendor_flag,weekend_invoice_flag
0,1,2,55,4,30,3,368659.03,0.1388,51169.87,2023,...,43311.65,70454120.26,3.571277,1,0,1,0,0,0,0
1,1,2,55,4,30,3,75588.68,0.1388,10491.71,2023,...,43311.65,70454120.26,0.732243,0,0,1,0,0,0,0
2,1,2,55,4,30,3,76126.01,0.1388,10566.29,2023,...,43311.65,70454120.26,0.737449,0,0,1,0,0,0,0
3,1,2,55,7,30,3,98334.87,0.1388,13648.88,2023,...,43311.65,70454120.26,0.952591,0,0,1,0,0,0,0
4,1,2,55,7,30,3,63983.73,0.1388,8880.94,2024,...,43311.65,70454120.26,0.619824,0,0,1,0,0,0,0


In [17]:
# Save the fully processed dataframe (features + target)
output_path = "processed_invoice_dataset.csv"
df_final.to_csv(output_path, index=False)

print(f"Success! Dataset saved to: {output_path}")

Success! Dataset saved to: processed_invoice_dataset.csv


In [18]:
# 1. Store counts
original_cols_count = 19
final_cols_count = df_final.shape[1]
new_features_created = final_cols_count - (original_cols_count - len(columns_to_drop))

# 2. Calculate target distribution (Fraud vs Non-Fraud ratio)
fraud_counts = y.value_counts(normalize=True) * 100

# 3. Calculate Memory Usage in Megabytes
memory_mb = df_final.memory_usage(deep=True).sum() / (1024 * 1024)

# 4. Print beautiful report
print("=" * 40)
print(" FEATURE ENGINEERING SUMMARY REPORT")
print("=" * 40)
print(f"Original Columns         : {original_cols_count}")
print(f"Columns Dropped          : {len(columns_to_drop)}")
print(f"New Features Created     : {new_features_created}")
print(f"Final Feature Count      : {X.shape[1]}")
print("-" * 40)
print("Target Distribution:")
print(f"Normal Invoices (0)      : {fraud_counts[0]:.2f}%")
print(f"Fraud Invoices (1)       : {fraud_counts[1]:.2f}%")
print("-" * 40)
print(f"Final Memory Usage       : {memory_mb:.2f} MB")
print("=" * 40)

 FEATURE ENGINEERING SUMMARY REPORT
Original Columns         : 19
Columns Dropped          : 10
New Features Created     : 45
Final Feature Count      : 53
----------------------------------------
Target Distribution:
Normal Invoices (0)      : 81.30%
Fraud Invoices (1)       : 18.70%
----------------------------------------
Final Memory Usage       : 5.80 MB
